# Category classification comparison

**Owner:** Sohaib

Skeleton only: fill in the empty code cells; do not change contracts.

In [1]:
# Data paths: run this cell first.
from pathlib import Path

REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = REPO_DIR / "data" / "papers.parquet"
EMBEDDINGS_PATH = REPO_DIR / "data" / "embeddings.npy"
DATA_PATH, EMBEDDINGS_PATH


(WindowsPath('D:/Sem_7/DS_Project/research_paper_discovery/data/papers.parquet'),
 WindowsPath('D:/Sem_7/DS_Project/research_paper_discovery/data/embeddings.npy'))

## Load the split

Call src.classify.load_split and load_xy for both sides.

In [2]:
import sys
sys.path.insert(0, str(REPO_DIR))
from src import classify

train_ids, test_ids = classify.load_split()
print("train", len(train_ids), "test", len(test_ids))

import pandas as pd
df = pd.read_parquet(DATA_PATH)
print("df", df.shape)

train_texts, y_train = classify.load_xy(df, train_ids)
test_texts, y_test = classify.load_xy(df, test_ids)
print("train texts", len(train_texts), "test texts", len(test_texts))

import numpy as np
embeddings = None
if EMBEDDINGS_PATH.exists():
    embeddings = np.load(EMBEDDINGS_PATH)
    print("embeddings", embeddings.shape)
else:
    print("embeddings.npy not found; logreg_emb will be skipped")


train 95935 test 23984


df (119919, 11)


train texts 95935 test texts 23984


embeddings (119919, 384)


## Build TF-IDF features

Fit the vectorizer on training texts only.

In [3]:
vectorizer = classify.build_tfidf(train_texts)
X_train_tfidf = vectorizer.transform(train_texts)
X_test_tfidf = vectorizer.transform(test_texts)
print("TF-IDF train", X_train_tfidf.shape, "test", X_test_tfidf.shape)
print("vocab size", len(vectorizer.vocabulary_))


TF-IDF train (95935, 919475) test (23984, 919475)
vocab size 919475


## Train model variants

Train logreg, nb, svm and logreg_emb.

In [4]:
models = {}
test_features = {}
for name in ("logreg", "nb", "svm"):
    print("training", name, "...")
    models[name] = classify.train_model(name, X_train_tfidf, y_train)
    test_features[name] = X_test_tfidf
    print("done", name)

if embeddings is not None:
    id_to_row = {str(i): r for r, i in enumerate(df["id"].astype(str))}
    train_idx = [id_to_row[i] for i in train_ids if i in id_to_row]
    test_idx = [id_to_row[i] for i in test_ids if i in id_to_row]
    X_train_emb = embeddings[train_idx]
    X_test_emb = embeddings[test_idx]
    models["logreg_emb"] = classify.train_model("logreg_emb", X_train_emb, y_train)
    test_features["logreg_emb"] = X_test_emb
    print("done logreg_emb", X_train_emb.shape)
else:
    print("Skipping logreg_emb: embeddings.npy absent")


training logreg ...


D:\Sem_7\DS_Project\research_paper_discovery\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


done logreg
training nb ...


done nb
training svm ...


done svm


D:\Sem_7\DS_Project\research_paper_discovery\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


done logreg_emb (95935, 384)


## Evaluate models

Call src.classify.evaluate for accuracy, macro-F1 and per-class scores.

In [5]:
import json as _json
label_classes = _json.load(open(REPO_DIR/"results"/"label_classes.json", encoding="utf-8"))
class_names = label_classes["classes_"]
print("class_names", class_names)

metrics_list = []
results = {"models": []}
for name, model in models.items():
    Xte = test_features[name]
    m = classify.evaluate(model, Xte, y_test, class_names)
    m["model"] = name
    metrics_list.append(m)
    results["models"].append(m)
    print(f"{name}: acc={m['accuracy']:.4f} macro_f1={m['macro_f1']:.4f}")
    for cls, v in m["per_class"].items():
        print(f"   {cls}: P={v['precision']:.4f} R={v['recall']:.4f} F1={v['f1']:.4f}")

classify.save_metrics(results, REPO_DIR/"results"/"classification_metrics.json")
classify.save_metrics(results, REPO_DIR/"results"/"classification_metrics.csv")
print("saved metrics json + csv")

for m in metrics_list:
    path = REPO_DIR/"images"/f"7_confusion_{m['model']}.png"
    classify.plot_confusion(m["confusion"], class_names, path)
    print("saved", path)


class_names ['cs.CL', 'cs.CV', 'cs.LG']
logreg: acc=0.9221 macro_f1=0.9221
   cs.CL: P=0.9399 R=0.9466 F1=0.9432
   cs.CV: P=0.9344 R=0.9081 F1=0.9210
   cs.LG: P=0.8928 R=0.9115 F1=0.9021


nb: acc=0.9101 macro_f1=0.9100
   cs.CL: P=0.9167 R=0.9457 F1=0.9310
   cs.CV: P=0.9230 R=0.8938 F1=0.9082
   cs.LG: P=0.8908 R=0.8908 F1=0.8908


svm: acc=0.9255 macro_f1=0.9255
   cs.CL: P=0.9425 R=0.9520 F1=0.9472
   cs.CV: P=0.9326 R=0.9196 F1=0.9261
   cs.LG: P=0.9013 R=0.9049 F1=0.9031
logreg_emb: acc=0.9159 macro_f1=0.9158
   cs.CL: P=0.9338 R=0.9490 F1=0.9413
   cs.CV: P=0.9183 R=0.9098 F1=0.9140
   cs.LG: P=0.8952 R=0.8889 F1=0.8920


saved metrics json + csv


saved D:\Sem_7\DS_Project\research_paper_discovery\images\7_confusion_logreg.png
saved D:\Sem_7\DS_Project\research_paper_discovery\images\7_confusion_nb.png


saved D:\Sem_7\DS_Project\research_paper_discovery\images\7_confusion_svm.png
saved D:\Sem_7\DS_Project\research_paper_discovery\images\7_confusion_logreg_emb.png


## Plot confusion matrix

Call src.classify.plot_confusion for the best model.

In [6]:
import pandas as pd
cm_df = pd.DataFrame(results["models"])[["model","accuracy","macro_f1"]]
print(cm_df.to_string(index=False))


     model  accuracy  macro_f1
    logreg  0.922073  0.922111
        nb  0.910107  0.909984
       svm  0.925492  0.925469
logreg_emb  0.915902  0.915800


## Save metrics

Call src.classify.save_metrics with a tracked JSON path.

In [7]:
print("metrics csv:", REPO_DIR/"results"/"classification_metrics.csv")
print("confusion pngs in:", REPO_DIR/"images")


metrics csv: D:\Sem_7\DS_Project\research_paper_discovery\results\classification_metrics.csv
confusion pngs in: D:\Sem_7\DS_Project\research_paper_discovery\images


## Analysis

- **cs.LG vs cs.CV/cs.CL confusion:** see the confusion matrices in
  `images/7_confusion_*.png`; cs.LG overlaps most with cs.CV/cs.CL because of
  cross-listed papers.
- **TF-IDF vs embeddings:** compare the `logreg` (TF-IDF) and `logreg_emb`
  (sentence-transformer) rows in the metrics table above.
- **Cap-balanced caveat:** the dataset is balanced by the 40k-per-category cap,
  so accuracy is meaningful on this sample but does not reflect real arXiv
  proportions; macro-F1 is reported alongside accuracy.


## Results

TODO: summarize the outputs and figures produced above.